In [1]:
!pip install "mlflow==2.22.4" --force-reinstall

  Using cached mlflow-2.22.4-py3-none-any.whl (29.0 MB)
  Using cached jinja2-3.1.6-py3-none-any.whl (134 kB)
  Using cached matplotlib-3.10.7-cp310-cp310-manylinux_2_27_aarch64.manylinux_2_28_aarch64.whl (9.5 MB)
  Using cached flask-3.1.2-py3-none-any.whl (103 kB)
  Using cached scipy-1.15.3-cp310-cp310-manylinux_2_17_aarch64.manylinux2014_aarch64.whl (35.5 MB)
  Using cached pyarrow-19.0.1-cp310-cp310-manylinux_2_28_aarch64.whl (40.5 MB)
  Using cached docker-7.1.0-py3-none-any.whl (147 kB)
  Using cached markdown-3.10-py3-none-any.whl (107 kB)
  Using cached mlflow_skinny-2.22.4-py3-none-any.whl (6.3 MB)
  Using cached graphene-3.4.3-py2.py3-none-any.whl (114 kB)
  Using cached sqlalchemy-2.0.44-cp310-cp310-manylinux_2_17_aarch64.manylinux2014_aarch64.whl (3.2 MB)
  Using cached scikit_learn-1.7.2-cp310-cp310-manylinux_2_27_aarch64.manylinux_2_28_aarch64.whl (9.5 MB)
  Using cached gunicorn-23.0.0-py3-none-any.whl (85 kB)
  Using cached pandas-2.3.3-cp310-cp310-manylinux_2_24_aarch

In [1]:
spark.sql("SHOW TABLES IN demo_db2").show(truncate=False)

+---------+------------------+-----------+
|namespace|tableName         |isTemporary|
+---------+------------------+-----------+
|demo_db2 |tb_user_churn_pred|false      |
|demo_db2 |tb_user_features  |false      |
|demo_db2 |tb_users          |false      |
+---------+------------------+-----------+



In [4]:
import os
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

from pyspark.sql import SparkSession

# Spark
spark = SparkSession.builder.appName("user-churn-train").getOrCreate()

# 1) MLflow Tracking
mlflow.set_tracking_uri("http://mlflow:5000")

# 2) MinIO / S3 环境
os.environ["MLFLOW_S3_ENDPOINT_URL"] = "http://minio:9000"
os.environ["AWS_ACCESS_KEY_ID"] = "admin"
os.environ["AWS_SECRET_ACCESS_KEY"] = "password"
os.environ["AWS_REGION"] = "us-east-1"

# 3) 实验
mlflow.set_experiment("demo-from-jupyter")

# 4) 从 Iceberg 特征表读取训练数据
feature_table = "demo_db.tb_user_features"
feature_date = "2025-12-06"   # 和 SQL 插入保持一致

sdf = (
    spark.table(feature_table)
         .where(f"feature_date = date('{feature_date}')")
)

# 转成 Pandas 方便给 sklearn 用
pdf = sdf.toPandas()

# 特征列 / 标签列（和 SQL 里字段名对应）
feature_cols = ["f_age", "f_is_vip", "f_city_idx"]
label_col = "label"

X = pdf[feature_cols].values
y = pdf[label_col].values

# 5) 训练 + MLflow 记录
with mlflow.start_run(run_name="rf-user-churn-train"):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=42
    )

    rf = RandomForestRegressor(
        n_estimators=100,
        max_depth=6,
        random_state=42
    )
    rf.fit(X_train, y_train)

    preds = rf.predict(X_test)
    mse = mean_squared_error(y_test, preds)
    rmse = mse ** 0.5

    # 记录参数 / 指标 / 元信息
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 6)
    mlflow.log_param("feature_table", feature_table)
    mlflow.log_param("feature_date", feature_date)
    mlflow.log_param("feature_cols", ",".join(feature_cols))

    mlflow.log_metric("rmse", rmse)

    # 记录模型
    mlflow.sklearn.log_model(rf, artifact_path="model")

print("done, rmse =", rmse)
print("训练使用特征表:", feature_table, "feature_date =", feature_date)

25/12/07 08:05:27 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.
2025/12/07 08:05:30 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run rf-user-churn-train at: http://mlflow:5000/#/experiments/1/runs/9c8617938f6b4992bd830e52c9e4e36c
🧪 View experiment at: http://mlflow:5000/#/experiments/1
done, rmse = 0.33376638536557274
训练使用特征表: demo_db.tb_user_features feature_date = 2025-12-06


In [8]:
import os
import mlflow
import mlflow.sklearn
from pyspark.sql import functions as F
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("user-churn-predict").getOrCreate()

# 1) MLflow 配置
mlflow.set_tracking_uri("http://mlflow:5000")

os.environ["MLFLOW_S3_ENDPOINT_URL"] = "http://minio:9000"
os.environ["AWS_ACCESS_KEY_ID"] = "admin"
os.environ["AWS_SECRET_ACCESS_KEY"] = "password"
os.environ["AWS_REGION"] = "us-east-1"

feature_table = "demo_db.tb_user_features"
feature_date = "2025-12-06"
feature_cols = ["f_age", "f_is_vip", "f_city_idx"]

target_table = "demo_db.tb_user_churn_pred"

with mlflow.start_run(run_name="rf-user-churn-predict"):

    # 2）加载已注册模型
    #    这里假设你在 UI 里注册的名字是 rf_user_churn，版本 1
    model_uri = "models:/rf_user_churn/1"
    model = mlflow.sklearn.load_model(model_uri)

    # 3）从 Iceberg 特征表读取本次需要预测的一批数据
    sdf_feat = (
        spark.table(feature_table)
             .where(F.col("feature_date") == F.to_date(F.lit(feature_date)))
    )

    pdf = sdf_feat.toPandas()
    X = pdf[feature_cols].values
    user_ids = pdf["user_id"].tolist()

    # 4）调用模型做预测
    preds = model.predict(X)

    # 当前预测 run 的 run_id，写回表里方便追溯
    run_id = mlflow.active_run().info.run_id

    # 5）构造结果 DataFrame
    rows = [
        (user_ids[i], float(preds[i]), "rf_user_churn", "1", run_id)
        for i in range(len(preds))
    ]

    pred_sdf = (
        spark.createDataFrame(
            rows,
            ["user_id", "score", "model_name", "model_version", "run_id"]
        )
        .withColumn("predict_time", F.current_timestamp())
    )

    # 6）写回预测结果表
    (
        pred_sdf
        .select(
            "user_id",
            "predict_time",
            "score",
            "model_name",
            "model_version",
            "run_id",
        )
        .writeTo(target_table)
        .append()
    )

    print("写回完成，记录数 =", pred_sdf.count())
    print("本次预测 run_id =", run_id)
    print("写入表 =", target_table)

写回完成，记录数 = 5
本次预测 run_id = c504870b1ac64b2bb37737f62a08808a
写入表 = demo_db.tb_user_churn_pred
🏃 View run rf-user-churn-predict at: http://mlflow:5000/#/experiments/1/runs/c504870b1ac64b2bb37737f62a08808a
🧪 View experiment at: http://mlflow:5000/#/experiments/1
